# 09 — Model Versioning
Save model to models/v1/ with metadata.json.

In [1]:

import pandas as pd
import numpy as np
import joblib
import json
import os
from datetime import datetime
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

PROC = r'../data/processed'
MODELS = r'../models'
V1_DIR = f'{MODELS}/v1'
os.makedirs(V1_DIR, exist_ok=True)

df = pd.read_csv(f'{PROC}/feature_matrix.csv')
target = 'AttritionRisk_Label'
drop_cols = [c for c in [target, 'EmployeeID'] if c in df.columns]
X = df.drop(columns=drop_cols).astype(float)
y = df[target]

pipeline = joblib.load(f'{MODELS}/attrition_pipeline.joblib')
print(f"Loaded pipeline: {type(pipeline.named_steps[list(pipeline.named_steps.keys())[-1]]).__name__}")


Loaded pipeline: XGBClassifier


In [2]:

# Re-evaluate for metadata
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    'precision': round(precision_score(y_test, y_pred), 4),
    'recall': round(recall_score(y_test, y_pred), 4),
    'f1': round(f1_score(y_test, y_pred), 4),
    'roc_auc': round(roc_auc_score(y_test, y_prob), 4)
}

# Read comparison to get winner name
clf_name = type(pipeline.named_steps[list(pipeline.named_steps.keys())[-1]]).__name__

metadata = {
    'version': 'v1',
    'model_name': clf_name,
    'trained_at': datetime.now().isoformat(),
    'training_data': 'employee_attrition_processed.csv',
    'feature_matrix': 'feature_matrix.csv',
    'n_features': X.shape[1],
    'n_training_samples': len(X_train),
    'n_test_samples': len(X_test),
    'class_balance': y.value_counts().to_dict(),
    'test_metrics': metrics,
    'selection_criterion': 'highest recall — missing flight-risk employee is expensive mistake',
    'hyperparameters': str(pipeline.get_params()),
    'feature_engineering': [
        'tenure_adjusted_salary: salary / (years_at_company + 1)',
        'years_since_last_promotion: 2024 - last_promotion_year',
        'overtime_to_projects_ratio: overtime_hours / (projects + 1)',
        'days_since_last_leave: days from last leave date',
        'engagement_score: standardized composite of WLB + Performance + CustomerSat'
    ],
    'retraining_triggers': [
        'data drift > threshold on age/salary/WLB/overtime/tenure',
        'F1 drops below 0.60',
        '6 months of new data collected'
    ]
}

print("=== Model v1 Metadata ===")
print(json.dumps({k:v for k,v in metadata.items() if k != 'hyperparameters'}, indent=2))


=== Model v1 Metadata ===
{
  "version": "v1",
  "model_name": "XGBClassifier",
  "trained_at": "2026-09-02T14:58:34.187283",
  "training_data": "employee_attrition_processed.csv",
  "feature_matrix": "feature_matrix.csv",
  "n_features": 43,
  "n_training_samples": 400,
  "n_test_samples": 100,
  "class_balance": {
    "0": 445,
    "1": 55
  },
  "test_metrics": {
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0,
    "roc_auc": 1.0
  },
  "selection_criterion": "highest recall \u2014 missing flight-risk employee is expensive mistake",
  "feature_engineering": [
    "tenure_adjusted_salary: salary / (years_at_company + 1)",
    "years_since_last_promotion: 2024 - last_promotion_year",
    "overtime_to_projects_ratio: overtime_hours / (projects + 1)",
    "days_since_last_leave: days from last leave date",
    "engagement_score: standardized composite of WLB + Performance + CustomerSat"
  ],
  "retraining_triggers": [
    "data drift > threshold on age/salary/WLB/overtime/tenure"

In [3]:

# Save
joblib.dump(pipeline, f'{V1_DIR}/attrition_pipeline.joblib')
with open(f'{V1_DIR}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved: {V1_DIR}/attrition_pipeline.joblib")
print(f"Saved: {V1_DIR}/metadata.json")
print(f"\nv1 Metrics: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, "
      f"F1={metrics['f1']:.4f}, ROC-AUC={metrics['roc_auc']:.4f}")


Saved: ../models/v1/attrition_pipeline.joblib
Saved: ../models/v1/metadata.json

v1 Metrics: Precision=1.0000, Recall=1.0000, F1=1.0000, ROC-AUC=1.0000


**Model versioning complete.** v1 model and metadata saved to models/v1/.